# Pipeline DLT Gold - Open-Meteo Clima

**Modelagem Dimensional - Dados Climáticos**

Análise de clima, qualidade do ar e previsões para capitais brasileiras.

## Dimensões
* dim_localidade - Capitais brasileiras com coordenadas geográficas
* dim_tempo_clima - Calendário temporal específico para séries climáticas

## Fatos
* fact_clima_historico - Dados históricos de temperatura, precipitação, vento
* fact_previsao_tempo - Previsões diárias de clima
* fact_qualidade_ar - Medições de poluentes atmosféricos e índices AQI

## Dimensão: Localidades

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW dim_localidade
(
  sk_localidade BIGINT COMMENT 'Chave substituta da localidade.',
  id BIGINT COMMENT 'ID único do geocoding.',
  capital STRING COMMENT 'Nome da capital consultada.',
  uf STRING COMMENT 'Sigla da unidade federativa.',
  nome_oficial STRING COMMENT 'Nome oficial da localidade segundo Open-Meteo.',
  latitude DOUBLE COMMENT 'Latitude em graus decimais.',
  longitude DOUBLE COMMENT 'Longitude em graus decimais.',
  elevation DOUBLE COMMENT 'Elevação em metros acima do nível do mar.',
  timezone STRING COMMENT 'Fuso horário completo (ex: America/Sao_Paulo).',
  populacao BIGINT COMMENT 'População estimada da localidade.',
  pais STRING COMMENT 'Nome do país.',
  codigo_pais STRING COMMENT 'Código ISO do país (BR).',
  CONSTRAINT not_null_sk_localidade EXPECT (sk_localidade IS NOT NULL) ON VIOLATION DROP ROW
)
COMMENT 'Dimensão de localidades (capitais brasileiras) com coordenadas e informações geográficas.'
TBLPROPERTIES (
  "quality" = "gold",
  pipelines.autoOptimize.managed = true,
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact = true
)
CLUSTER BY AUTO
AS
SELECT DISTINCT
  xxhash64(
    COALESCE(CAST(cl.id AS STRING), ''),
    COALESCE(cl.capital, ''),
    COALESCE(cl.uf, '')
  ) AS sk_localidade,
  cl.id,
  cl.capital,
  cl.uf,
  cl.name AS nome_oficial,
  cl.latitude,
  cl.longitude,
  cl.elevation,
  CONCAT(cl.timezone_continente, '/', cl.timezone_estado) AS timezone,
  cl.population AS populacao,
  cl.country AS pais,
  cl.country_code AS codigo_pais
FROM silver_${var.environment}.ds_open_meteo.cleaned_geocoding_localidades cl
WHERE cl.id IS NOT NULL;

## Dimensão: Tempo Clima

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW dim_tempo_clima
(
  sk_data INT COMMENT 'Chave substituta da data no formato YYYYMMDD.',
  data DATE COMMENT 'Data completa.',
  ano INT COMMENT 'Ano.',
  mes INT COMMENT 'Mês (1-12).',
  trimestre INT COMMENT 'Trimestre (1-4).',
  semestre INT COMMENT 'Semestre (1-2).',
  nome_mes STRING COMMENT 'Nome do mês.',
  dia_ano INT COMMENT 'Dia do ano (1-366).',
  semana_ano INT COMMENT 'Semana do ano (1-53).',
  dia_semana INT COMMENT 'Dia da semana (1=Domingo).',
  nome_dia_semana STRING COMMENT 'Nome do dia da semana.',
  eh_fim_semana BOOLEAN COMMENT 'Indicador de fim de semana.',
  estacao STRING COMMENT 'Estação do ano (Verão, Outono, Inverno, Primavera).',
  CONSTRAINT not_null_sk_data EXPECT (sk_data IS NOT NULL) ON VIOLATION DROP ROW
)
COMMENT 'Dimensão de calendário temporal para análises climáticas e sazonalidade.'
TBLPROPERTIES (
  "quality" = "gold",
  pipelines.autoOptimize.managed = true,
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact = true
)
CLUSTER BY AUTO
AS
WITH datas_unicas AS (
  SELECT DISTINCT time AS data
  FROM silver_${var.environment}.ds_open_meteo.cleaned_air_quality
  WHERE time IS NOT NULL
  
  UNION
  
  SELECT DISTINCT time AS data
  FROM silver_${var.environment}.ds_open_meteo.cleaned_daily_forecast
  WHERE time IS NOT NULL
  
  UNION
  
  SELECT DISTINCT DATE(time) AS data
  FROM silver_${var.environment}.ds_open_meteo.cleaned_historical_weather
  WHERE time IS NOT NULL
)
SELECT
  CAST(DATE_FORMAT(data, 'yyyyMMdd') AS INT) AS sk_data,
  data,
  YEAR(data) AS ano,
  MONTH(data) AS mes,
  QUARTER(data) AS trimestre,
  CASE WHEN MONTH(data) <= 6 THEN 1 ELSE 2 END AS semestre,
  CASE MONTH(data)
    WHEN 1 THEN 'Janeiro' WHEN 2 THEN 'Fevereiro' WHEN 3 THEN 'Março'
    WHEN 4 THEN 'Abril' WHEN 5 THEN 'Maio' WHEN 6 THEN 'Junho'
    WHEN 7 THEN 'Julho' WHEN 8 THEN 'Agosto' WHEN 9 THEN 'Setembro'
    WHEN 10 THEN 'Outubro' WHEN 11 THEN 'Novembro' WHEN 12 THEN 'Dezembro'
  END AS nome_mes,
  DAYOFYEAR(data) AS dia_ano,
  WEEKOFYEAR(data) AS semana_ano,
  DAYOFWEEK(data) AS dia_semana,
  CASE DAYOFWEEK(data)
    WHEN 1 THEN 'Domingo' WHEN 2 THEN 'Segunda-feira' WHEN 3 THEN 'Terça-feira'
    WHEN 4 THEN 'Quarta-feira' WHEN 5 THEN 'Quinta-feira' WHEN 6 THEN 'Sexta-feira'
    WHEN 7 THEN 'Sábado'
  END AS nome_dia_semana,
  CASE WHEN DAYOFWEEK(data) IN (1, 7) THEN true ELSE false END AS eh_fim_semana,
  CASE 
    WHEN MONTH(data) IN (12, 1, 2) THEN 'Verão'
    WHEN MONTH(data) IN (3, 4, 5) THEN 'Outono'
    WHEN MONTH(data) IN (6, 7, 8) THEN 'Inverno'
    ELSE 'Primavera'
  END AS estacao
FROM datas_unicas;

## Fato: Clima Histórico

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW fact_clima_historico
(
  sk_localidade BIGINT COMMENT 'Chave estrangeira para dim_localidade.',
  sk_data INT COMMENT 'Chave estrangeira para dim_tempo_clima.',
  data_hora TIMESTAMP COMMENT 'Data e hora completa da medição.',
  temperatura_2m DOUBLE COMMENT 'Temperatura a 2 metros do solo em °C.',
  temperatura_aparente DOUBLE COMMENT 'Temperatura aparente (sensação térmica) em °C.',
  precipitacao DOUBLE COMMENT 'Precipitação em mm.',
  chuva DOUBLE COMMENT 'Chuva em mm.',
  velocidade_vento_10m DOUBLE COMMENT 'Velocidade do vento a 10m em km/h.',
  direcao_vento_10m DOUBLE COMMENT 'Direção do vento em graus (0-360).',
  umidade_relativa_2m DOUBLE COMMENT 'Umidade relativa do ar a 2m em %.',
  pressao_nivel_mar DOUBLE COMMENT 'Pressão atmosférica ao nível do mar em hPa.'
)
COMMENT 'Tabela fato de dados climáticos históricos horários para capitais brasileiras.'
TBLPROPERTIES (
  "quality" = "gold",
  pipelines.autoOptimize.managed = true,
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact = true
)
CLUSTER BY AUTO
AS
SELECT
  xxhash64(
    COALESCE(chw.capital, ''),
    COALESCE(chw.uf, '')
  ) AS sk_localidade,
  CAST(DATE_FORMAT(DATE(chw.time), 'yyyyMMdd') AS INT) AS sk_data,
  chw.time AS data_hora,
  chw.temperature_2m AS temperatura_2m,
  chw.apparent_temperature AS temperatura_aparente,
  chw.precipitation AS precipitacao,
  chw.rain AS chuva,
  chw.wind_speed_10m AS velocidade_vento_10m,
  chw.wind_direction_10m AS direcao_vento_10m,
  chw.relative_humidity_2m AS umidade_relativa_2m,
  chw.pressure_msl AS pressao_nivel_mar
FROM silver_${var.environment}.ds_open_meteo.cleaned_historical_weather chw
WHERE chw.time IS NOT NULL;

## Fato: Previsão do Tempo

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW fact_previsao_tempo
(
  sk_localidade BIGINT COMMENT 'Chave estrangeira para dim_localidade.',
  sk_data INT COMMENT 'Chave estrangeira para dim_tempo_clima.',
  temperatura_max DOUBLE COMMENT 'Temperatura máxima prevista em °C.',
  temperatura_min DOUBLE COMMENT 'Temperatura mínima prevista em °C.',
  temperatura_media DOUBLE COMMENT 'Temperatura média prevista em °C.',
  precipitacao_total DOUBLE COMMENT 'Precipitação total prevista em mm.',
  velocidade_vento_max DOUBLE COMMENT 'Velocidade máxima do vento prevista em km/h.',
  codigo_clima STRING COMMENT 'Código WMO do tipo de clima.',
  nascer_sol TIMESTAMP COMMENT 'Horário do nascer do sol.',
  por_sol TIMESTAMP COMMENT 'Horário do pôr do sol.',
  tempo_geracao_ms DOUBLE COMMENT 'Tempo de geração da previsão em ms.'
)
COMMENT 'Tabela fato de previsões diárias de tempo para capitais brasileiras.'
TBLPROPERTIES (
  "quality" = "gold",
  pipelines.autoOptimize.managed = true,
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact = true
)
CLUSTER BY AUTO
AS
SELECT
  xxhash64(
    COALESCE(CAST(cdf.latitude AS STRING), ''),
    COALESCE(CAST(cdf.longitude AS STRING), '')
  ) AS sk_localidade,
  CAST(DATE_FORMAT(cdf.time, 'yyyyMMdd') AS INT) AS sk_data,
  cdf.temperature_2m_max AS temperatura_max,
  cdf.temperature_2m_min AS temperatura_min,
  cdf.temperature_2m_mean AS temperatura_media,
  cdf.precipitation_sum AS precipitacao_total,
  cdf.wind_speed_10m_max AS velocidade_vento_max,
  CAST(cdf.weather_code AS STRING) AS codigo_clima,
  cdf.sunrise AS nascer_sol,
  cdf.sunset AS por_sol,
  cdf.generationtime_ms AS tempo_geracao_ms
FROM silver_${var.environment}.ds_open_meteo.cleaned_daily_forecast cdf
WHERE cdf.time IS NOT NULL;

## Fato: Qualidade do Ar

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW fact_qualidade_ar
(
  sk_localidade BIGINT COMMENT 'Chave estrangeira para dim_localidade.',
  sk_data INT COMMENT 'Chave estrangeira para dim_tempo_clima.',
  data_hora TIMESTAMP COMMENT 'Data e hora da medição.',
  monoxido_carbono DOUBLE COMMENT 'Concentração de CO em µg/m³.',
  dioxido_nitrogenio DOUBLE COMMENT 'Concentração de NO₂ em µg/m³.',
  ozonio DOUBLE COMMENT 'Concentração de O₃ em µg/m³.',
  pm2_5 DOUBLE COMMENT 'Material particulado PM2.5 em µg/m³.',
  pm10 DOUBLE COMMENT 'Material particulado PM10 em µg/m³.',
  dioxido_enxofre DOUBLE COMMENT 'Concentração de SO₂ em µg/m³.',
  indice_europeu_aqi INT COMMENT 'Índice de qualidade do ar europeu (1-5).',
  indice_us_aqi INT COMMENT 'Índice de qualidade do ar dos EUA (0-500).',
  indice_uv DOUBLE COMMENT 'Índice de radiação ultravioleta.'
)
COMMENT 'Tabela fato de medições de qualidade do ar e poluentes atmosféricos.'
TBLPROPERTIES (
  "quality" = "gold",
  pipelines.autoOptimize.managed = true,
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact = true
)
CLUSTER BY AUTO
AS
SELECT
  xxhash64(
    COALESCE(caq.capital, ''),
    COALESCE(caq.uf, '')
  ) AS sk_localidade,
  CAST(DATE_FORMAT(DATE(caq.time), 'yyyyMMdd') AS INT) AS sk_data,
  caq.time AS data_hora,
  caq.carbon_monoxide AS monoxido_carbono,
  caq.nitrogen_dioxide AS dioxido_nitrogenio,
  caq.ozone AS ozonio,
  caq.pm2_5,
  caq.pm10,
  caq.sulphur_dioxide AS dioxido_enxofre,
  caq.european_aqi AS indice_europeu_aqi,
  caq.us_aqi AS indice_us_aqi,
  caq.uv_index AS indice_uv
FROM silver_${var.environment}.ds_open_meteo.cleaned_air_quality caq
WHERE caq.time IS NOT NULL;